In [ ]:
import os
from contextlib import contextmanager

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from spikingjelly.activation_based import neuron, surrogate, functional, layer, monitor

from src.cams.gradcam import SNNGradCAM
from src.cams.ablation_cam import AblationCAM
from src.cams.mc_ablation_cam import MCAblationCAM
from src.cams.sam import SpikeActivationMap
from src.cams.guided_backprop import GuidedBackpropCAM

from utils.args_utils import setup_config_dictionary
from src.utils.data_utils import get_dataloaders
from src.utils.model_utils import get_model
from src.utils.save_utils import interactive_load_model, load_model
from src.utils.plot_utils import plot_frames, plot_cam_time_series, combine_polarities, plot_aggregated_cam

from plot_landscape import plot_activation_landscape

# Repro + device
print(torch.seed())
# torch.manual_seed(1937388963656961197)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = setup_config_dictionary()
config["DATASET_NAME"] = "dvs_gesture"
config["MODEL_NAME"] = "simple_snn_no_vote"
config["TAU"] = 2.0
config["N_TIME_BINS"] = 16
# the number of time steps must be set from the config file, since the dataloader reads it from there

print('Device:', device)
print('Dataset:', config['DATASET_NAME'])
print('Model:', config['MODEL_NAME'])

In [ ]:
train_loader, test_loader = get_dataloaders(
    config['DATASET_NAME'],
    batch_size=1,
    n_time_bins=config["N_TIME_BINS"],
)

sample_events, sample_labels = next(iter(train_loader))
# encoder = PoissonEncoder(step_mode='m')
encoder = None
was_single_image = False

if sample_events.dim() == 4:
    was_single_image = True
    sample_events = sample_events.unsqueeze(1)  # add time dimension
    sample_events = sample_events.repeat(1, config["N_TIME_BINS"], 1, 1, 1)  # repeat along time dimension

if encoder is not None:
    sample_events = encoder(sample_events)

sample_events = sample_events.transpose(0, 1)

T, B, C, H, W = sample_events.shape
print('Events Shape (T, B, C, H, W):', (T, B, C, H, W))

num_classes = len(train_loader.dataset.classes) if hasattr(train_loader.dataset, "classes") else len(train_loader.dataset.dataset.classes) if hasattr(train_loader.dataset, "dataset") else len(train_loader.dataset.base_dataset.classes)
spiking_neuron = neuron.LIFNode if not was_single_image else neuron.IFNode

print("The spiking neuron used is:", spiking_neuron)

params = {
    "model_name": config["MODEL_NAME"],
    "num_classes": num_classes,
    "in_channels": C,
    "spiking_neuron": spiking_neuron,
    "pretrained": False,
    "detach_reset": False,
    "surrogate_function": surrogate.ATan(),
}
if not was_single_image:
    params["tau"] = config["TAU"]
model = get_model(**params)

functional.set_step_mode(model, 'm')
functional.set_backend(model, "torch", instance=spiking_neuron)
model.to(device)

# uncomment to pick which model to load, otherwise just run load_model, which loads the latest
# interactive_load_model(model, config['MODEL_SAVE_DIR'], config['DATASET_NAME'])
load_model(model, config["MODEL_SAVE_DIR"], config["DATASET_NAME"])

model.eval()
print("Model loaded.")
print(f"Model is {model}")


gesture_map = {
    0: "hand_clapping",
    1: "right_hand_wave",
    2: "left_hand_wave",
    3: "right_hand_clockwise",
    4: "right_hand_counter_clockwise",
    5: "left_hand_clockwise",
    6: "left_hand_counter_clockwise",
    7: "forearm_roll_forward",
    8: "forearm_roll_backward",
    9: "drums",
    10: "guitar",
    #11: "random_other_gestures", # -> not implemented here
}

In [ ]:
def benchmark_sparsity(net, dataloader, device='cuda', ignore_final_layer=False):
    net.eval()
    net.to(device)
    
    # We will track the total number of spikes and total neuron-steps across the whole dataset
    total_spikes = 0
    total_elements = 0
    
    # 1. Register a monitor to record output spikes of every spiking neuron layer
    #    OutputMonitor automatically records the output of layers you specify
    spike_monitor = monitor.OutputMonitor(net, neuron.LIFNode)
    
    with torch.no_grad():
        for img, label in dataloader:
            if img.dim() == 4:
                img = img.unsqueeze(1)  # add time dimension
                img = img.repeat(1, config["N_TIME_BINS"], 1, 1, 1)  # repeat along time dimension
    
            img = img.transpose(0, 1).to(device)

            img = img.to(device)
            
            # Forward pass
            functional.reset_net(net)
            
            # If your input is static, you might repeat it over T timesteps here
            # For this snippet, assuming img is already (T, B, C, H, W) or handled by the net
            net(img)
            
            # 2. Iterate through recorded spike tensors to calculate sparsity
            # spike_monitor.records is a list of tensors, one for each monitored layer
            records = spike_monitor.records
            if ignore_final_layer and len(records) > 0:
                records = records[:-1]

            for spike_tensor in records:
                # spike_tensor shape: [T, Batch, ...]
                total_spikes += spike_tensor.sum().item()
                total_elements += spike_tensor.numel()
            spike_monitor.clear_recorded_data()

    # 3. Calculate Global Average Firing Rate
    avg_firing_rate = total_spikes / total_elements
    print(f"Global Average Firing Rate (Sparsity): {avg_firing_rate:.4f}")
    
    # Optional: Breakdown by layer?
    # You would need to store totals per layer index instead of a global sum.
    
    return avg_firing_rate

avg_firing_rate = benchmark_sparsity(model, train_loader, device=device, ignore_final_layer=True)

In [ ]:
K_TRIALS = 20
CHANNEL_BATCH_SIZE = 32 # Depending on input size and GPU memory, this can be increased for faster computation
def cam_variance_T_HW(cam):
    if isinstance(cam, torch.Tensor):
        cam = cam.detach().cpu().numpy()
    return float(cam.var(axis=(1, 2)).mean())

@contextmanager
def grad_cam_context(model):
    """
    Context manager to temporarily enable gradients and set training mode 
    (preserving BN/Dropout eval state) for GradCAM.
    """
    prev_training = model.training   
    # 1. Enable Grads
    for p in model.parameters():
        p.requires_grad = True 
    # 2. Set Train mode (needed for SNN surrogate grads)
    model.train()
    # 3. Freeze BN/Dropout (we don't want to update stats or drop units during interpretation)
    for m in model.modules():
        if isinstance(m, (layer.BatchNorm1d, layer.BatchNorm2d, layer.BatchNorm3d, layer.Dropout, layer.Dropout2d)):
            m.eval()   
    try:
        yield
    finally:
        # 4. Cleanup: Zero grads, disable grads, restore mode, reset state
        model.zero_grad()
        for p in model.parameters():
            p.requires_grad = False
        model.train(prev_training)
        functional.reset_net(model)

def get_conv_layers(m):
    return [x for x in m.modules() if isinstance(x, layer.Conv2d)]

def get_lif_layers(m):
    return [x for x in m.modules() if isinstance(x, (neuron.LIFNode, neuron.IFNode))]

def generate_grad_cam(events, class_idx, target_layer_idx, model):
    with grad_cam_context(model):
        target = get_conv_layers(model)[target_layer_idx]
        grad_cam = SNNGradCAM(model, target)
        return grad_cam.generate(events, class_idx=class_idx).detach()

def generate_ablation_cam(events, class_idx, target_layer_idx, model, use_pre_hook=False):
    functional.reset_net(model)
    target = get_conv_layers(model)[target_layer_idx]
    ab_cam = AblationCAM(model, target, channel_batch_size=CHANNEL_BATCH_SIZE, use_pre_hook=use_pre_hook)
    return ab_cam.generate(events, class_idx=class_idx).detach()

def generate_mc_ablation_cam(events, class_idx, target_layer_idx, model, use_post_hook=False):
    functional.reset_net(model)
    target = get_lif_layers(model)[target_layer_idx]
    mc_cam = MCAblationCAM(model, target, channel_batch_size=CHANNEL_BATCH_SIZE, k_trials=K_TRIALS, use_post_hook=use_post_hook)
    return mc_cam.generate(events, class_idx=class_idx).detach(), mc_cam # slightly different return

def generate_sam(events, target_layer_idx, model):
    functional.reset_net(model)
    target = get_lif_layers(model)[target_layer_idx]
    sam = SpikeActivationMap(model, target)
    return sam.generate(events).detach()

def generate_guided_backprop_cam(events, class_idx, model):
    with grad_cam_context(model):
        guided_backprop_cam = GuidedBackpropCAM(model)
        return guided_backprop_cam.generate(events, class_idx=class_idx, upsample=False).detach()

In [ ]:
conv_layers = get_conv_layers(model)
lif_layers = get_lif_layers(model)
print(f'Found {len(conv_layers)} conv layers and {len(lif_layers)} (L)IF layers.')
target_layer_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

for events, labels in test_loader:
    functional.reset_net(model)
    was_single_image = False
    if events.dim() == 4:
        was_single_image = True
        events = events.unsqueeze(1)  # add time dimension
        events = events.repeat(1, config["N_TIME_BINS"], 1, 1, 1)  # repeat along time dimension
    
    pre_encoding_events = events.clone()

    events = events.transpose(0, 1).to(device)
    label = labels.to(device)
    
    # Determine Ground Truth Classes
    target_classes = []
    if label.dim() == 2 and label.shape[1] > 1:
        # Multi-hot encoding (B, C)
        target_classes = torch.nonzero(label[0]).flatten().tolist()
    else:
        # Index encoding (B,) or (B, 1)
        target_classes = [int(label[0].item())]

    print(f'\nGround Truth Label(s): {target_classes}')
    if "dvs_gesture" in config["DATASET_NAME"]:
        print(f"Gesture Name(s): {[gesture_map.get(c, 'Unknown') for c in target_classes]}")
    outputs = model(events)  # (T, B, num_classes)
    time_sum = outputs.sum(dim=0)
    # print the outputs per class if there are less than 20 classes
    if num_classes <= 20:
        print(f"Outputs per class over time: {time_sum[0].detach().cpu().numpy()}")
    
    # Determine which GT classes were correctly identified
    # For single label, check if argmax matches.
    # For multi-label, check if GT class is in top K (e.g. 5).
    is_multilabel = (len(target_classes) > 1) or ("voc" in config["DATASET_NAME"])
    k = 5 if is_multilabel else 1
    top_preds = time_sum[0].topk(min(k, num_classes)).indices.tolist()
    
    valid_classes = [t for t in target_classes if t in top_preds]
    
    if not valid_classes:
        print(f"Skipping CAM generation. No GT classes {target_classes} found in top-{k} predictions {top_preds}.")
        continue

    print(f"Generating CAMs for correctly identified classes: {valid_classes}")

    frames = pre_encoding_events[0, :].detach().cpu()  # (T, C, H, W)
    if frames.shape[1] == 2:  # (T, 2, H, W) -> (T, 1, H, W)
        combined = combine_polarities(frames.permute(0,2,3,1).numpy())  # (T,H,W)
        combined = np.expand_dims(combined, axis=1)
    else: # 3 channels, assume RGB
        combined = frames.numpy()

    plot_frames(combined)

    for target_class in valid_classes:
        print(f"\n=== Generating CAMs for Class {target_class} ===")
        if "dvs_gesture" in config["DATASET_NAME"]:
            print(f"Gesture Name: {gesture_map.get(target_class, 'Unknown')}")
        confidence = F.softmax(time_sum, dim=1)[0, target_class].item() if num_classes > 1 else F.sigmoid(time_sum)[0, target_class].item()
        print(f'Class {target_class} confidence: {confidence:.4f}')

        mc_var_by_layer = []
        naive_var_by_layer = []
        sam_var_by_layer = []
        grad_var_by_layer = []
        for layer_index in target_layer_indices:
            if layer_index >= len(conv_layers) or layer_index >= len(lif_layers):
                break
            print(f'Targeting layer index {layer_index} for CAMs, out of {len(conv_layers)} conv layers and {len(lif_layers)} LIF layers.')
            # Grad-CAM
            grad_cams = generate_grad_cam(events, target_class, layer_index, model)
            print("Grad-CAMs over time:")
            if was_single_image: plot_aggregated_cam(combined, grad_cams)
            # print(f"combined shape: {combined.shape}, grad_cams shape: {grad_cams.shape}")
            plot_cam_time_series(combined, grad_cams, per_frame_cam_norm=False, images_per_row=8)

            # Monte Carlo Ablation CAM
            mc_ablation_cams, mc_ablation_cam_obj = generate_mc_ablation_cam(events, target_class, layer_index, model)
            print("MC Ablation CAMs over time:")
            if was_single_image: plot_aggregated_cam(combined, mc_ablation_cams)
            plot_cam_time_series(combined, mc_ablation_cams, per_frame_cam_norm=False, images_per_row=8)
            
            # # Monte Carlo Ablation CAM post-hook
            # mc_ablation_cams, mc_ablation_cam_obj = generate_mc_ablation_cam(events, target_class, layer_index, model, use_post_hook=True)
            # print("MC Ablation CAMs with post hook over time:")
            # if was_single_image: plot_aggregated_cam(combined, mc_ablation_cams)
            # plot_cam_time_series(combined, mc_ablation_cams, per_frame_cam_norm=False, images_per_row=8)

            # Ablation CAM
            ab_cams = generate_ablation_cam(events, target_class, layer_index, model)
            print("Ablation CAMs over time:")
            if was_single_image: plot_aggregated_cam(combined, ab_cams)
            plot_cam_time_series(combined, ab_cams, per_frame_cam_norm=False, images_per_row=8)

            # # Pre-hook Ablation CAM
            # prehook_ab_cams = generate_ablation_cam(events, target_class, layer_index, model, use_pre_hook=True)
            # print("Pre-hook Ablation CAMs over time:")
            # if was_single_image: plot_aggregated_cam(combined, prehook_ab_cams)
            # plot_cam_time_series(combined, prehook_ab_cams, per_frame_cam_norm=False, images_per_row=8)

            # Spike Activation Map
            sams = generate_sam(events, layer_index, model)
            print("SAMs over time:")
            if was_single_image: plot_aggregated_cam(combined, sams)
            plot_cam_time_series(combined, sams, per_frame_cam_norm=False, images_per_row=8)

            # # Guided Backprop CAM
            # gb_cams = generate_guided_backprop_cam(events, target_class, model)
            # print("Guided Backprop CAMs over time:")
            # if was_single_image: plot_aggregated_cam(combined, gb_cams)
            # plot_cam_time_series(combined, gb_cams, per_frame_cam_norm=False, images_per_row=8)

            baseline_conv = mc_ablation_cam_obj.convergence_data['baseline']
            x_axis = range(1, K_TRIALS + 1)
            ch_10_conv = mc_ablation_cam_obj.convergence_data['ch_10']
            plt.plot(x_axis, baseline_conv, label="Baseline Score")
            plt.plot(x_axis, ch_10_conv, label="Channel 10 Ablated Score")
            plt.xlabel("Number of Trials (k)")
            plt.ylabel("Average Score")
            plt.legend()
            plt.title("MC Ablation CAM (post-hook) Score Convergence")
            plt.show()

            mc_var_by_layer.append(cam_variance_T_HW(mc_ablation_cams))
            naive_var_by_layer.append(cam_variance_T_HW(ab_cams))
            sam_var_by_layer.append(cam_variance_T_HW(sams))
            grad_var_by_layer.append(cam_variance_T_HW(grad_cams))
            print("Done processing for this layer.")

            
        # plot activation landscape for the last targeted layer
        plot_activation_landscape(model, target_layer_indices[-1], events, target_class)
        plt.figure(figsize=(8, 4))
        xs = list(range(len(grad_var_by_layer)))
        plt.plot(xs, mc_var_by_layer, marker="o", label="MC Ablation CAM")
        plt.plot(xs, naive_var_by_layer, marker="o", label="Ablation CAM")
        plt.plot(xs, sam_var_by_layer, marker="o", label="SAM")
        plt.plot(xs, grad_var_by_layer, marker="o", label="Grad-CAM")
        plt.xlabel("Layer index")
        plt.ylabel("Mean pixel variance (avg over T)")
        plt.title(f"CAM spatial variance across layers (Class {target_class})")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()
    
    break  # only do one sample

In [ ]:

# Configuration
v_thr = 1.0
# Range of membrane potentials to evaluate (from below 0 to above threshold)
v_mem = np.linspace(-0.5, 1.5, 1000)

# --- 1. Standard Deterministic Behavior ---
# A neuron fires 1.0 if potential >= threshold, else 0.0
response_deterministic = (v_mem >= v_thr).astype(float)

# --- 2. MC Ablation (Noisy) Behavior ---
# Logic from your code: (inp[0] + torch.rand_like(inp[0]) * v_thr)
# We simulate K trials to show the "Average" (Expected Value)
k_trials = 10000

# Shape: (k_trials, len(v_mem))
# Uniform noise U(0, v_thr)
noise = np.random.rand(k_trials, len(v_mem)) * v_thr

# Add noise to potential
v_noisy = v_mem + noise

# Check firing condition against original threshold
spikes_noisy = (v_noisy >= v_thr).astype(float)

# Calculate probability of firing (Average over K trials)
response_probabilistic = spikes_noisy.mean(axis=0)

# --- Plotting ---
plt.figure(figsize=(10, 6))

# Plot Standard Step Function
plt.step(v_mem, response_deterministic, where='post', color='black', linestyle='--', 
            linewidth=2, label='Standard LIF (Hard Threshold)')

# Plot Probabilistic Curve
plt.plot(v_mem, response_probabilistic, color='#d62728', linewidth=3, alpha=0.9, 
            label=f'MC Ablation (Avg of {k_trials} trials)')

# Highlight the smoothing region
plt.fill_between(v_mem, response_deterministic, response_probabilistic, 
                    color='#d62728', alpha=0.1, label='Gradient Flow Area')

# Annotations
plt.title(f'Effect of Uniform Noise on Spiking Activation\nLogic: $V_{{noisy}} = V_{{mem}} + U(0, V_{{thr}})$', fontsize=14)
plt.xlabel('Input Membrane Potential ($V_{mem}$)', fontsize=12)
plt.ylabel('Output / Firing Probability', fontsize=12)

# Axis markers
plt.axvline(x=v_thr, color='gray', linestyle=':', alpha=0.6, label='$V_{thr}$')
plt.axvline(x=0, color='gray', linestyle=':', alpha=0.6)
plt.text(v_thr + 0.02, 0.5, 'Threshold', rotation=90, verticalalignment='center', color='gray')

plt.legend(loc='upper left', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.show()

